# Customer Segmentation using Mall Customers-style Data

This notebook demonstrates how to segment customers using K-Means clustering, identify the best number of groups with the elbow method, and turn cluster profiles into practical marketing actions.

We will:
- load a customer dataset similar to the Mall Customers dataset
- clean and validate the data
- scale the features for clustering
- choose the number of clusters using the elbow method
- apply K-Means and visualize clusters
- profile each segment by age and spending behavior
- save cluster labels and a segment summary report

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Load a Mall Customers-like dataset or create a realistic fallback dataset if the CSV is not available.
try:
    df = pd.read_csv('Mall_Customers.csv')
except FileNotFoundError:
    np.random.seed(42)
    n = 200
    df = pd.DataFrame({
        'CustomerID': np.arange(1, n + 1),
        'Gender': np.random.choice(['Male', 'Female'], size=n),
        'Age': np.random.randint(18, 70, size=n),
        'Annual Income (k$)': np.random.randint(15, 137, size=n),
        'Spending Score (1-100)': np.random.randint(1, 101, size=n)
    })

    # Add visible customer groups to make clustering meaningful.
    df.loc[:60, 'Annual Income (k$)'] += 40
    df.loc[:60, 'Spending Score (1-100)'] += 25

    df.loc[61:120, 'Annual Income (k$)'] += 15
    df.loc[61:120, 'Spending Score (1-100)'] += 5

    df.loc[121:, 'Annual Income (k$)'] -= 20
    df.loc[121:, 'Spending Score (1-100)'] -= 15

    df['Annual Income (k$)'] = df['Annual Income (k$)'].clip(lower=10)
    df['Spending Score (1-100)'] = df['Spending Score (1-100)'].clip(lower=1, upper=100)

# Keep the columns that matter for segmentation.
selected_cols = ['CustomerID', 'Gender', 'Age', 'Annual Income (k$)', 'Spending Score (1-100)']
df = df[selected_cols].copy()

df.head()

## 1. Clean the dataset

We check for missing values, invalid numbers, and duplicate rows before segmentation. This ensures the clustering is based on reliable customer attributes.

In [ ]:
# Clean missing values and fix numeric columns.
df = df.drop_duplicates().reset_index(drop=True)
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['Annual Income (k$)'] = pd.to_numeric(df['Annual Income (k$)'], errors='coerce')
df['Spending Score (1-100)'] = pd.to_numeric(df['Spending Score (1-100)'], errors='coerce')

df = df.dropna().reset_index(drop=True)

print(f'Rows after cleaning: {len(df)}')
df.describe().round(2)

In [ ]:
# Explore the main customer attributes.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(df['Age'], bins=15, color='steelblue', edgecolor='black')
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Customers')

axes[1].hist(df['Spending Score (1-100)'], bins=15, color='darkorange', edgecolor='black')
axes[1].set_title('Spending Score Distribution')
axes[1].set_xlabel('Spending Score')
axes[1].set_ylabel('Customers')

plt.tight_layout()
plt.show()

## 2. Scale the features

K-Means is sensitive to feature scale, so we standardize the numerical attributes before computing distances.

In [ ]:
features = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
X = df[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled[:5]

In [ ]:
# Elbow method to choose the number of clusters.
inertia = []
for k in range(1, 11):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertia.append(model.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), inertia, marker='o', color='navy')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Within-Cluster Sum of Squares')
plt.grid(True)
plt.show()

# Use the elbow point as the selected number of clusters.
selected_k = 4
selected_k

## 3. Apply K-Means clustering

We fit K-Means using the elbow-selected number of clusters and assign each customer to a segment.

In [ ]:
model = KMeans(n_clusters=selected_k, random_state=42, n_init=20)
df['Cluster'] = model.fit_predict(X_scaled)
df['Cluster'] = df['Cluster'].astype(int)

df.head()

In [ ]:
# Profile each cluster by age, income, and spending.
cluster_profile = (df.groupby('Cluster')
    .agg(
        Customers=('CustomerID', 'count'),
        Avg_Age=('Age', 'mean'),
        Avg_Income=('Annual Income (k$)', 'mean'),
        Avg_Spending=('Spending Score (1-100)', 'mean')
    )
    .round(2)
    .sort_values('Avg_Spending', ascending=False))

cluster_profile

In [ ]:
# Cluster visualization: income vs spending score.
plt.figure(figsize=(10, 6))
plt.scatter(
    df['Annual Income (k$)'],
    df['Spending Score (1-100)'],
    c=df['Cluster'],
    cmap='viridis',
    s=90,
    alpha=0.8,
    edgecolors='k',
    linewidth=0.5
)
plt.title('Customer Segments: Income vs Spending Score')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.colorbar(label='Cluster')
plt.show()

# Optional second view: age vs spending score.
plt.figure(figsize=(10, 6))
plt.scatter(
    df['Age'],
    df['Spending Score (1-100)'],
    c=df['Cluster'],
    cmap='plasma',
    s=90,
    alpha=0.8,
    edgecolors='k',
    linewidth=0.5
)
plt.title('Customer Segments: Age vs Spending Score')
plt.xlabel('Age')
plt.ylabel('Spending Score (1-100)')
plt.colorbar(label='Cluster')
plt.show()

## 4. Derive marketing actions from each cluster

A good customer segmentation model should separate groups with different value and behavior patterns, so each segment can receive tailored marketing actions.

In [ ]:
# Create a short business description for each cluster.
segment_labels = {}
for cluster_id, row in cluster_profile.iterrows():
    if row['Avg_Spending'] >= 75 and row['Avg_Income'] >= 75:
        label = 'High-value loyal customers'
        action = 'Offer premium loyalty perks, exclusive products, and early access promotions.'
    elif row['Avg_Spending'] >= 60 and row['Avg_Income'] < 60:
        label = 'Budget-conscious premium shoppers'
        action = 'Target with value bundles, cashback offers, and discount campaigns.'
    elif row['Avg_Spending'] < 45 and row['Avg_Income'] >= 60:
        label = 'Potential churn risk'
        action = 'Send re-engagement offers and personalized recommendations to increase spend.'
    else:
        label = 'Moderate value customers'
        action = 'Use targeted email campaigns, cross-sell bundles, and loyalty reminders.'
    
    segment_labels[cluster_id] = label
    print(f'Cluster {cluster_id}: {label} | Action: {action}')

# Save cluster labels for downstream analysis.
df['Segment_Label'] = df['Cluster'].map(segment_labels)
df.to_csv('customer_segments.csv', index=False)

segment_summary = cluster_profile.copy()
segment_summary['Segment_Label'] = segment_summary.index.map(segment_labels)
segment_summary['Marketing_Action'] = segment_summary.index.map({
    k: (
        'Offer premium loyalty perks, exclusive products, and early access promotions.' if v == 'High-value loyal customers' else
        'Target with value bundles, cashback offers, and discount campaigns.' if v == 'Budget-conscious premium shoppers' else
        'Send re-engagement offers and personalized recommendations to increase spend.' if v == 'Potential churn risk' else
        'Use targeted email campaigns, cross-sell bundles, and loyalty reminders.'
    )
    for k, v in segment_labels.items()
})

segment_summary = segment_summary[['Segment_Label', 'Customers', 'Avg_Age', 'Avg_Income', 'Avg_Spending', 'Marketing_Action']]
segment_summary.to_csv('segment_report.csv', index=True)
segment_summary

## 5. Final takeaway

This segmentation workflow gives us a practical view of customer groups based on age, income, and spending behavior. The model can be used with a live customer database to improve targeting, personalize offers, and prioritize retention campaigns.

- Save the cluster labels as: customer_segments.csv
- Save the segment report as: segment_report.csv